In [6]:
import sentencepiece as spm

import random
import os

TARGET_SIZE = 200000
TEMP_SKZ = "corpus/train_for_sp.skz.temp"
TEMP_ZH = "corpus/train_for_sp.zh.temp"

# Load Endfield data (guaranteed full inclusion)
with open("corpus/endfield.skz", "r", encoding="utf-8") as f:
    ef_skz = f.readlines()
with open("corpus/endfield.zh", "r", encoding="utf-8") as f:
    ef_zh = f.readlines()

# Load main aligned corpus
with open("corpus/train.skz", "r", encoding="utf-8") as f:
    train_skz = f.readlines()
with open("corpus/train.zh", "r", encoding="utf-8") as f:
    train_zh = f.readlines()

# Calculate remaining slots for the main corpus
remaining_slots = max(0, TARGET_SIZE - len(ef_skz))

# Randomly sample from main corpus if needed
if remaining_slots > 0 and len(train_skz) > remaining_slots:
    indices = random.sample(range(len(train_skz)), remaining_slots)
    sample_skz = [train_skz[i] for i in indices]
    sample_zh = [train_zh[i] for i in indices]
else:
    sample_skz = train_skz
    sample_zh = train_zh

# Combine: Endfield first, then sampled main corpus
combined_skz = ef_skz + sample_skz
combined_zh = ef_zh + sample_zh

# Write to temporary files
with open(TEMP_SKZ, "w", encoding="utf-8") as f_skz, \
     open(TEMP_ZH, "w", encoding="utf-8") as f_zh:
    f_skz.writelines(combined_skz)
    f_zh.writelines(combined_zh)

print(f"Temporary corpus created: {len(combined_skz)} lines")
print(f"  Endfield included: {len(ef_skz)} lines")
print(f"  Sampled from main: {len(sample_skz)} lines")

Temporary corpus created: 200000 lines
  Endfield included: 13096 lines
  Sampled from main: 186904 lines


## Train Sarkaz Tokenizer 

In [7]:
print("Training Sarkaz tokenizer on prioritized temporary corpus...")
spm.SentencePieceTrainer.Train(
    input=TEMP_SKZ,
    model_prefix="models/sp_merged_skz",
    vocab_size=128,
    character_coverage=1.0,
    model_type="unigram",
    # shuffle_input_sentence=False,  # Preserves our manual priority order
    shuffle_input_sentence=True, 
    split_by_whitespace=False,
    split_digits=False,
    num_threads=8
)
print("Sarkaz tokenizer training complete.")

Training Sarkaz tokenizer on prioritized temporary corpus...
Sarkaz tokenizer training complete.


## Train Chinese Tokenizer

In [8]:
print("Training Chinese tokenizer on prioritized temporary corpus...")
spm.SentencePieceTrainer.Train(
    input=TEMP_ZH,
    model_prefix="models/sp_merged_zh",
    vocab_size=16000,
    character_coverage=0.9995,
    model_type="unigram",
    # shuffle_input_sentence=False,  # Preserves our manual priority order
    shuffle_input_sentence=True, 
    max_sentencepiece_length=16,
    num_threads=8
)
print("Chinese tokenizer training complete.")

Training Chinese tokenizer on prioritized temporary corpus...
Chinese tokenizer training complete.


## Clean Up

In [9]:
temp_files = [TEMP_SKZ, TEMP_ZH]
for path in temp_files:
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed temporary file: {path}")

print("Cleanup complete. Tokenizers are ready for training.")

Removed temporary file: corpus/train_for_sp.skz.temp
Removed temporary file: corpus/train_for_sp.zh.temp
Cleanup complete. Tokenizers are ready for training.


## Coverage Validation

In [10]:
def check_coverage(model_path, corpus_path):
    sp = spm.SentencePieceProcessor()
    sp.Load(model_path)
    
    with open(corpus_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
    total_tokens = 0
    unk_count = 0
    
    for line in lines:
        tokens = sp.encode(line.strip(), out_type=str)
        total_tokens += len(tokens)
        unk_count += tokens.count("<unk>")
        
    ratio = unk_count / max(total_tokens, 1)
    print(f"Corpus: {corpus_path} | Vocab: {model_path}")
    print(f"  Total tokens: {total_tokens} | <unk> count: {unk_count} | Ratio: {ratio:.4%}")
    
    if ratio > 0.005:
        print("  WARNING: High <unk> ratio detected. Consider increasing vocab_size or checking preprocessing consistency.")
    else:
        print("  OK: Coverage is sufficient for training.")

print("Running coverage validation on validation set...")
check_coverage("models/sp_merged_skz.model", "corpus/val.skz")
check_coverage("models/sp_merged_zh.model", "corpus/val.zh")

Running coverage validation on validation set...
Corpus: corpus/val.skz | Vocab: models/sp_merged_skz.model
  Total tokens: 34202569 | <unk> count: 0 | Ratio: 0.0000%
  OK: Coverage is sufficient for training.
Corpus: corpus/val.zh | Vocab: models/sp_merged_zh.model
  Total tokens: 27963684 | <unk> count: 0 | Ratio: 0.0000%
  OK: Coverage is sufficient for training.
